# Notebook de Previsão Semanal de Crimes - PrePol

Este notebook gera previsões de probabilidade de crimes para a próxima semana usando o modelo RandomForest treinado.

## 🎯 Objetivo
Prever probabilidades de ocorrência de crimes em células H3 para a próxima semana com base em dados históricos.

## ⚙️ Otimizações de Performance do Mapa
- **Max Zoom:** 19 (permite zoom detalhado)
- **Prefer Canvas:** Renderização otimizada
- **Smooth Factor:** Melhora qualidade visual em diferentes níveis de zoom
- **Filtro de Probabilidade:** Células com > 5% de probabilidade (ajustável)

## 💡 Dicas para Melhor Performance
Se o mapa estiver lento ou não carregar:
1. **Aumente o filtro de probabilidade** na célula do mapa (ex: > 10% ou > 15%)
2. **Reduza o número de células** filtrando por área geográfica específica
3. **Salve o mapa em HTML** para visualização externa: `m.save('mapa.html')`

## 1. Setup

In [12]:
# Import libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json
import folium
import time
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add prepol module to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import prepol.config as config
import prepol.helpers as helpers

pd.set_option('display.max_columns', None)

print("✓ Libraries loaded")
print(f"✓ Project root: {project_root}")

✓ Libraries loaded
✓ Project root: d:\BackupSupremo\Work\prepol-project


## 2. Load Model

In [13]:
# Load trained model
# CONFIGURE MODEL: Set to None for latest, or specify filename
model_filename = "rf_crime_model_20251125_1448.joblib"  # e.g., "rf_crime_model_20251125_1448.joblib"

model_dir = project_root / "model"

if model_filename:
    # Load specific model
    model_path = model_dir / model_filename
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")
else:
    # Load latest model
    model_files = sorted(model_dir.glob(f"{config.MODEL_PREFIX}_*.joblib"))
    if not model_files:
        raise FileNotFoundError(f"No model found in {model_dir}")
    model_path = model_files[-1]

meta_path = model_path.with_name(model_path.stem.replace(config.MODEL_PREFIX, f"{config.MODEL_PREFIX}_meta") + ".json")

print(f"Loading model: {model_path.name}")

# Load model and metadata
rf_model = joblib.load(model_path)
with open(meta_path, 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_columns']

print(f"✓ Model loaded")
print(f"  Test R²: {metadata['metrics']['test']['r2']:.4f}")
print(f"  Features: {feature_cols}")

Loading model: rf_crime_model_20251125_1448.joblib
✓ Model loaded
  Test R²: 0.9324
  Features: ['RUBRICA', 'crime_type_encoded', 'y_norm', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_rol_3', 'y_rol_7', 'y_lag_1_vizinhos']


## 3. Load Panel Data

In [14]:
# Load panel data
# UPDATE THIS PATH to point to your panel file
panel_filename = "PrePol_Panel_25_11_2025_by_crimetype_weekly.parquet"  # Change this as needed
panel_path = Path.cwd() / "prepol_out" / panel_filename

print(f"Loading panel data from: {panel_path.name}")
df_panel = pd.read_parquet(panel_path)

# Convert Period to string for easier handling
if 'time_period' in df_panel.columns:
    if pd.api.types.is_period_dtype(df_panel['time_period']):
        df_panel['time_period_str'] = df_panel['time_period'].astype(str)
    else:
        df_panel['time_period_str'] = df_panel['time_period']

print(f"✓ Panel loaded: {df_panel.shape}")
print(f"  Date range: {df_panel['time_period'].min()} to {df_panel['time_period'].max()}")
print(f"  H3 cells: {df_panel['h3_cell'].nunique():,}")
print(f"  Time periods: {df_panel['time_period_str'].nunique()}")

df_panel.head()

Loading panel data from: PrePol_Panel_25_11_2025_by_crimetype_weekly.parquet
✓ Panel loaded: (40487945, 13)
  Date range: 2297 to 2453
✓ Panel loaded: (40487945, 13)
  Date range: 2297 to 2453
  H3 cells: 51,577
  H3 cells: 51,577
  Time periods: 157
  Time periods: 157


,h3_cell,time_period,RUBRICA,crime_type_encoded,y,y_norm,y_lag_1,y_lag_2,y_lag_3,y_rol_3,y_rol_7,y_lag_1_vizinhos,time_period_str
0,8aa810000007fff,2297,Furto (art. 155),0,0,-0.113230,NaN,NaN,NaN,NaN,NaN,NaN,2297
1,8aa810000007fff,2298,Furto (art. 155),0,0,-0.113230,0.0,NaN,NaN,0.0,0.0,NaN,2298
2,8aa810000007fff,2299,Furto (art. 155),0,0,-0.113230,0.0,0.0,NaN,0.0,0.0,NaN,2299
3,8aa810000007fff,2300,Furto (art. 155),0,0,-0.113230,0.0,0.0,0.0,0.0,0.0,NaN,2300
4,8aa810000007fff,2301,Furto (art. 155),0,1,8.775327,0.0,0.0,0.0,0.0,0.0,NaN,2301


## 4. Generate Future Date Range & Synthetic Panel

In [15]:
# Generate next week from last available date in panel
print("🔮 Generating future forecast period...")

# Get the last available period from panel
last_panel_period = df_panel['time_period'].max()
print(f"  Last available panel period: {last_panel_period}")
print(f"  Period type: {type(last_panel_period)}")

# Check if time_period is stored as integer or Period
if isinstance(last_panel_period, (int, np.integer)):
    # Period is stored as integer (ordinal)
    print("  Note: time_period stored as integer ordinal")
    
    # Generate next period (simply increment the integer)
    forecast_period_ordinal = int(last_panel_period) + 1
    
    # For display, we'll use the ordinal value
    # We'll reconstruct the actual date range from the panel's last time_period
    if 'time_period' in df_panel.columns:
        last_timestamp = df_panel[df_panel['time_period'] == last_panel_period]['time_period'].max()
        # Forecast starts after last time_period (next week)
        forecast_start = pd.to_datetime(last_timestamp) + timedelta(days=7)
    else:
        # If no time_period, estimate from time_period ordinal
        # Assuming weekly periods starting from some base date
        forecast_start = pd.Timestamp('2013-01-01') + pd.Timedelta(weeks=forecast_period_ordinal)
    
    forecast_end = forecast_start + timedelta(days=6)
    forecast_period = forecast_period_ordinal
    
else:
    # Period is stored as pandas Period object
    forecast_period = last_panel_period + 1
    forecast_start = forecast_period.start_time
    forecast_end = forecast_period.end_time

target_start = forecast_start.strftime('%Y-%m-%d')
target_end = forecast_end.strftime('%Y-%m-%d')
target_period = f"{target_start} to {target_end}"

print(f"\n🎯 Forecast period: {target_period}")
print(f"   Period value: {forecast_period}")
print(f"   Duration: {(forecast_end - forecast_start).days + 1} days")

# Get all unique H3 cells from existing panel
unique_cells = df_panel['h3_cell'].unique()
print(f"\n📊 Building synthetic panel...")
print(f"  Unique H3 cells: {len(unique_cells):,}")
print(f"  Forecast periods: 1 week")
print(f"  Total rows to generate: {len(unique_cells):,}")

# Build synthetic panel data for future week
synthetic_rows = []

# Get last known values for each cell to use for lag features
print(f"  Extracting last known features for each cell...")
last_known = df_panel.sort_values('time_period').groupby('h3_cell').last().reset_index()

# Identify all columns that should be carried forward
base_cols = ['h3_cell', 'time_period', 'time_period_str', 'time_period', 'y']
lag_cols = ['y_norm', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_rol_3', 'y_rol_7', 'y_lag_1_vizinhos']

for cell in unique_cells:
    # Get last known features for this cell
    cell_data = last_known[last_known['h3_cell'] == cell]
    
    if len(cell_data) == 0:
        # Cell has no history - use zeros for all features
        row = {
            'h3_cell': cell,
            'time_period': forecast_period,
            'time_period_str': str(forecast_period),
            'time_period': forecast_start,
            'y': 0,  # No actual crimes (future)
        }
        # Add lag features as zeros
        for col in lag_cols:
            row[col] = 0
        # Add any other columns from panel as zeros/defaults
        for col in df_panel.columns:
            if col not in row and col not in ['time_period', 'time_period_str']:
                row[col] = 0
    else:
        # Use last known features from historical data
        cell_last = cell_data.iloc[0]
        row = {
            'h3_cell': cell,
            'time_period': forecast_period,
            'time_period_str': str(forecast_period),
            'time_period': forecast_start,
            'y': 0,  # No actual crimes (future)
            'y_norm': cell_last.get('y_norm', 0),
            'y_lag_1': cell_last.get('y', 0),  # Last known actual becomes lag_1
            'y_lag_2': cell_last.get('y_lag_1', 0),
            'y_lag_3': cell_last.get('y_lag_2', 0),
            'y_rol_3': cell_last.get('y_rol_3', 0),
            'y_rol_7': cell_last.get('y_rol_7', 0),
            'y_lag_1_vizinhos': cell_last.get('y_lag_1_vizinhos', 0),
        }
        
        # Carry forward any additional columns from the panel (like RUBRICA, crime_type_encoded)
        for col in df_panel.columns:
            if col not in row and col not in ['time_period', 'time_period_str', 'y']:
                row[col] = cell_last.get(col, 0)
    
    synthetic_rows.append(row)

# Create synthetic panel dataframe
df_target = pd.DataFrame(synthetic_rows)

print(f"\n✓ Synthetic panel created: {len(df_target):,} records")
print(f"  Unique cells: {df_target['h3_cell'].nunique():,}")
print(f"  Forecast period: {df_target['time_period'].iloc[0]}")
print(f"  Columns: {list(df_target.columns)}")

# Preview feature statistics
print(f"\n📈 Feature Statistics (from last known data):")
print(f"  • y_norm: mean={df_target['y_norm'].mean():.3f}, max={df_target['y_norm'].max():.3f}")
print(f"  • y_lag_1: mean={df_target['y_lag_1'].mean():.3f}, max={df_target['y_lag_1'].max():.1f}")
print(f"  • y_rol_7: mean={df_target['y_rol_7'].mean():.3f}, max={df_target['y_rol_7'].max():.3f}")

df_target.head()

🔮 Generating future forecast period...
  Last available panel period: 2453
  Period type: <class 'numpy.int64'>
  Note: time_period stored as integer ordinal

🎯 Forecast period: 1970-01-08 to 1970-01-14
   Period value: 2454
   Duration: 7 days

📊 Building synthetic panel...
  Unique H3 cells: 51,577
  Forecast periods: 1 week
  Total rows to generate: 51,577
  Extracting last known features for each cell...

🎯 Forecast period: 1970-01-08 to 1970-01-14
   Period value: 2454
   Duration: 7 days

📊 Building synthetic panel...
  Unique H3 cells: 51,577
  Forecast periods: 1 week
  Total rows to generate: 51,577
  Extracting last known features for each cell...

✓ Synthetic panel created: 51,577 records
  Unique cells: 51,577
  Forecast period: 1970-01-08 00:00:00.000002453
  Columns: ['h3_cell', 'time_period', 'time_period_str', 'y', 'y_norm', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_rol_3', 'y_rol_7', 'y_lag_1_vizinhos', 'RUBRICA', 'crime_type_encoded']

📈 Feature Statistics (from last known 

,h3_cell,time_period,time_period_str,y,y_norm,y_lag_1,y_lag_2,y_lag_3,y_rol_3,y_rol_7,y_lag_1_vizinhos,RUBRICA,crime_type_encoded
0,8aa810000007fff,1970-01-08 00:00:00.000002453,2454,0,-0.259988,0,1.0,0.0,0.333333,0.142857,NaN,Roubo (art. 157),4
1,8aa81000000ffff,1970-01-08 00:00:00.000002453,2454,0,-0.139127,0,0.0,0.0,0.000000,0.000000,NaN,Roubo (art. 157),4
2,8aa810000017fff,1970-01-08 00:00:00.000002453,2454,0,0.000000,0,0.0,0.0,0.000000,0.000000,NaN,OUTROS,3
3,8aa81000001ffff,1970-01-08 00:00:00.000002453,2454,0,-0.113230,0,0.0,0.0,0.000000,0.000000,NaN,Furto (art. 155),0
4,8aa810000027fff,1970-01-08 00:00:00.000002453,2454,0,0.000000,0,0.0,0.0,0.000000,0.000000,NaN,OUTROS,3


## 5. Generate Predictions

In [16]:
# Prepare features for prediction on synthetic future data
print("🔮 Generating predictions for future week...")
df_prep = df_target.copy()

# Convert Period to ordinal (required for model)
for col in df_prep.columns:
    if pd.api.types.is_period_dtype(df_prep[col]):
        df_prep[col] = df_prep[col].apply(lambda x: x.ordinal if pd.notna(x) else np.nan)

# Ensure all feature columns exist and are numeric
print(f"\n📋 Preparing features: {feature_cols}")
missing_features = [col for col in feature_cols if col not in df_prep.columns]
if missing_features:
    print(f"  ⚠️  Missing features (will be filled with 0): {missing_features}")
    for col in missing_features:
        df_prep[col] = 0

# Handle categorical columns if they exist as strings
for col in feature_cols:
    if col in df_prep.columns:
        # If column is object type (string), try to convert to numeric
        if df_prep[col].dtype == 'object':
            print(f"  🔄 Converting categorical column '{col}' to numeric...")
            # Use factorize to convert strings to numeric codes
            df_prep[col] = pd.factorize(df_prep[col])[0]

# Extract features and predict
X_target = df_prep[feature_cols].fillna(0)

# Verify all features are numeric
print(f"\n✓ Feature matrix prepared: {X_target.shape}")
print(f"  Data types: {X_target.dtypes.value_counts().to_dict()}")

y_pred = rf_model.predict(X_target)

# Add predictions to dataframe
df_target['y_pred'] = np.clip(y_pred, 0, None)  # No negative predictions

print(f"\n✓ Predictions generated for {len(df_target):,} cells")
print(f"  Total predicted crimes (1 week): {df_target['y_pred'].sum():.0f}")
print(f"  Average per cell: {df_target['y_pred'].mean():.3f}")
print(f"  Cells with predictions > 0.5/week: {(df_target['y_pred'] > 0.5).sum():,}")


🔮 Generating predictions for future week...

📋 Preparing features: ['RUBRICA', 'crime_type_encoded', 'y_norm', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_rol_3', 'y_rol_7', 'y_lag_1_vizinhos']
  🔄 Converting categorical column 'RUBRICA' to numeric...

✓ Feature matrix prepared: (51577, 9)
  Data types: {dtype('float64'): 5, dtype('int64'): 2, dtype('float32'): 1, dtype('int8'): 1}

✓ Predictions generated for 51,577 cells
  Total predicted crimes (1 week): 1412
  Average per cell: 0.027
  Cells with predictions > 0.5/week: 1,090


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [17]:
# Prepare predictions for visualization
# Convert weekly predictions to daily averages (to match CrimeMap.jsx and server.py)

print("Preparing predictions for visualization...")

# POISSON probability conversion function (matches server.py)
def count_to_probability(daily_avg_count):
    """Convert daily average crime count to probability using Poisson distribution.
    
    P(≥1 crime on a day) = 1 - P(0 crimes) = 1 - e^(-λ)
    where λ = daily average crime count
    
    This is the scientifically correct approach for count data and matches
    the backend API implementation in server.py and CrimeMap.jsx.
    
    Args:
        daily_avg_count: Predicted daily average crime count
    
    Returns:
        Probability between 0 and 1 of at least one crime occurring on a day
    """
    return 1 - np.exp(-daily_avg_count)

# Add coordinates to df_target (cell centroid for each H3 cell)
print("Adding H3 cell coordinates...")
df_target['lat'] = df_target['h3_cell'].apply(lambda x: helpers.h3_to_geo(x)[0])
df_target['lon'] = df_target['h3_cell'].apply(lambda x: helpers.h3_to_geo(x)[1])

# Convert weekly predictions to daily averages (to match server.py behavior)
df_target['predicted_daily_avg'] = df_target['y_pred'] / 7.0
df_target['n_days'] = 7  # Weekly forecast = 7 days
df_target['predicted_total'] = df_target['y_pred']  # Keep weekly total

# Calculate probability with POISSON distribution on DAILY average (matches server.py)
df_target['crime_probability'] = df_target['predicted_daily_avg'].apply(count_to_probability)
df_target['normalized_prob'] = df_target['crime_probability']

# Use df_target directly for visualization (no aggregation needed for weekly data)
df_cell_agg = df_target.copy()

# Distribution analysis
print(f"✓ Prepared {len(df_cell_agg):,} cells for visualization")
print(f"  Total predicted crimes (1 week): {df_cell_agg['predicted_total'].sum():.0f}")
print(f"  Mean predicted daily avg: {df_cell_agg['predicted_daily_avg'].mean():.3f}")
print(f"  Median predicted daily avg: {df_cell_agg['predicted_daily_avg'].median():.3f}")

print(f"\n📊 Probability Distribution (Poisson on Daily Avg):")
print(f"  • Range: {df_cell_agg['crime_probability'].min()*100:.1f}% to {df_cell_agg['crime_probability'].max()*100:.1f}%")
print(f"  • Mean: {df_cell_agg['crime_probability'].mean()*100:.1f}%")
print(f"  • Median: {df_cell_agg['crime_probability'].median()*100:.1f}%")
print(f"  • Cells with <20% probability: {(df_cell_agg['crime_probability'] < 0.2).sum():,}")
print(f"  • Cells with 20-50% probability: {((df_cell_agg['crime_probability'] >= 0.2) & (df_cell_agg['crime_probability'] < 0.5)).sum():,}")
print(f"  • Cells with 50-80% probability: {((df_cell_agg['crime_probability'] >= 0.5) & (df_cell_agg['crime_probability'] < 0.8)).sum():,}")
print(f"  • Cells with >80% probability: {(df_cell_agg['crime_probability'] >= 0.8).sum():,}")

print(f"\n🎯 Weekly Predictions:")
print(f"  • Min daily avg: {df_cell_agg['predicted_daily_avg'].min():.4f}")
print(f"  • 25th percentile: {df_cell_agg['predicted_daily_avg'].quantile(0.25):.4f}")
print(f"  • Median: {df_cell_agg['predicted_daily_avg'].quantile(0.5):.4f}")
print(f"  • 75th percentile: {df_cell_agg['predicted_daily_avg'].quantile(0.75):.4f}")
print(f"  • Max daily avg: {df_cell_agg['predicted_daily_avg'].max():.4f}")

print(f"\n✓ Probability calculation matches server.py and CrimeMap.jsx")

df_cell_agg.head()


Preparing predictions for visualization...
Adding H3 cell coordinates...
✓ Prepared 51,577 cells for visualization
  Total predicted crimes (1 week): 1412
  Mean predicted daily avg: 0.004
  Median predicted daily avg: 0.000

📊 Probability Distribution (Poisson on Daily Avg):
  • Range: 0.0% to 82.7%
  • Mean: 0.3%
  • Median: 0.0%
  • Cells with <20% probability: 51,448
  • Cells with 20-50% probability: 119
  • Cells with 50-80% probability: 9
  • Cells with >80% probability: 1

🎯 Weekly Predictions:
  • Min daily avg: 0.0000
  • 25th percentile: 0.0000
  • Median: 0.0000
  • 75th percentile: 0.0000
  • Max daily avg: 1.7551

✓ Probability calculation matches server.py and CrimeMap.jsx
✓ Prepared 51,577 cells for visualization
  Total predicted crimes (1 week): 1412
  Mean predicted daily avg: 0.004
  Median predicted daily avg: 0.000

📊 Probability Distribution (Poisson on Daily Avg):
  • Range: 0.0% to 82.7%
  • Mean: 0.3%
  • Median: 0.0%
  • Cells with <20% probability: 51,448
  

,h3_cell,time_period,time_period_str,y,y_norm,y_lag_1,y_lag_2,y_lag_3,y_rol_3,y_rol_7,y_lag_1_vizinhos,RUBRICA,crime_type_encoded,y_pred,lat,lon,predicted_daily_avg,n_days,predicted_total,crime_probability,normalized_prob
0,8aa810000007fff,1970-01-08 00:00:00.000002453,2454,0,-0.259988,0,1.0,0.0,0.333333,0.142857,NaN,Roubo (art. 157),4,0.000121,-23.709493,-46.633485,1.735696e-05,7,0.000121,1.735681e-05,1.735681e-05
1,8aa81000000ffff,1970-01-08 00:00:00.000002453,2454,0,-0.139127,0,0.0,0.0,0.000000,0.000000,NaN,Roubo (art. 157),4,0.000018,-23.708578,-46.634333,2.503328e-06,7,0.000018,2.503325e-06,2.503325e-06
2,8aa810000017fff,1970-01-08 00:00:00.000002453,2454,0,0.000000,0,0.0,0.0,0.000000,0.000000,NaN,OUTROS,3,0.000001,-23.709335,-46.632168,1.513458e-07,7,0.000001,1.513458e-07,1.513458e-07
3,8aa81000001ffff,1970-01-08 00:00:00.000002453,2454,0,-0.113230,0,0.0,0.0,0.000000,0.000000,NaN,Furto (art. 155),0,0.000011,-23.708420,-46.633016,1.567339e-06,7,0.000011,1.567338e-06,1.567338e-06
4,8aa810000027fff,1970-01-08 00:00:00.000002453,2454,0,0.000000,0,0.0,0.0,0.000000,0.000000,NaN,OUTROS,3,0.000001,-23.710566,-46.633954,1.513458e-07,7,0.000001,1.513458e-07,1.513458e-07


## 6. Create Interactive Map

In [18]:
# Export predicted panel to sparse parquet format
print("💾 Exporting predictions to parquet...")

# Create prediction panel (weekly data - one row per cell)
# IMPORTANT: Include RUBRICA for crime type filtering
columns_to_export = ['h3_cell', 'time_period', 'predicted_daily_avg', 'y_pred', 'crime_probability', 'lat', 'lon', 'n_days']

# Add RUBRICA if it exists in df_cell_agg (for crime type filtering)
if 'RUBRICA' in df_cell_agg.columns:
    columns_to_export.append('RUBRICA')
    print("  ✓ Including RUBRICA column for crime type filtering")

df_predicted_panel = df_cell_agg[columns_to_export].copy()
df_predicted_panel.rename(columns={
    'time_period': 'period',
    'y_pred': 'predicted_total'
}, inplace=True)

# Generate timestamp for filename
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = Path.cwd() / "prepol_out" / f"prepol_forecast_{timestamp}.parquet"

# Ensure output directory exists
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save as parquet (sparse format)
df_predicted_panel.to_parquet(output_path, index=False, compression='snappy')

print(f"✓ Predictions exported: {output_path.name}")
print(f"  Total rows: {len(df_predicted_panel):,}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"  Cells: {df_predicted_panel['h3_cell'].nunique():,}")
print(f"  Forecast period: {forecast_period}")
print(f"  Columns: {list(df_predicted_panel.columns)}")

# Save summary metadata (matches CrimeMap.jsx statistics)
summary = {
    'forecast_generated': datetime.now().isoformat(),
    'forecast_period': {
        'period': str(forecast_period),
        'start': target_start,
        'end': target_end,
        'frequency': 'weekly',
        'n_days': 7
    },
    'statistics': {
        'total_cells': int(len(df_cell_agg)),
        'displayed_cells': int(len(df_cell_agg[df_cell_agg['crime_probability'] > 0.05])),
        'mean_probability': float(df_cell_agg['crime_probability'].mean()),
        'median_probability': float(df_cell_agg['crime_probability'].median()),
        'mean_daily_avg': float(df_cell_agg['predicted_daily_avg'].mean()),
        'total_predicted_crimes': float(df_cell_agg['predicted_total'].sum()),
        'high_risk_cells_80pct': int((df_cell_agg['crime_probability'] > 0.8).sum()),
        'high_risk_cells_50pct': int((df_cell_agg['crime_probability'] > 0.5).sum())
    },
    'model': {
        'name': model_path.name,
        'test_r2': metadata['metrics']['test']['r2']
    },
    'method': {
        'probability_model': 'Poisson (P >= 1 crime/day)',
        'conversion': 'weekly_predictions / 7 = daily_avg',
        'matches': 'server.py + CrimeMap.jsx'
    },
    'output_file': output_path.name
}

summary_path = output_path.with_suffix('.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Summary saved: {summary_path.name}")
print(f"✓ Export format matches CrimeMap.jsx data expectations")


💾 Exporting predictions to parquet...
  ✓ Including RUBRICA column for crime type filtering
✓ Predictions exported: prepol_forecast_20251127_212251.parquet
  Total rows: 51,577
  File size: 1634.5 KB
  Cells: 51,577
  Forecast period: 2454
  Columns: ['h3_cell', 'period', 'predicted_daily_avg', 'predicted_total', 'crime_probability', 'lat', 'lon', 'n_days', 'RUBRICA']
✓ Summary saved: prepol_forecast_20251127_212251.json
✓ Export format matches CrimeMap.jsx data expectations
✓ Predictions exported: prepol_forecast_20251127_212251.parquet
  Total rows: 51,577
  File size: 1634.5 KB
  Cells: 51,577
  Forecast period: 2454
  Columns: ['h3_cell', 'period', 'predicted_daily_avg', 'predicted_total', 'crime_probability', 'lat', 'lon', 'n_days', 'RUBRICA']
✓ Summary saved: prepol_forecast_20251127_212251.json
✓ Export format matches CrimeMap.jsx data expectations


In [19]:
# Prepare map data for forecast visualization
print("Preparing forecast map data...")

# Use aggregated cell data
df_map_source = df_cell_agg.copy()

print(f"✓ Using aggregated cell data: {len(df_map_source):,} unique cells")
print(f"  Average daily prediction: {df_map_source['y_pred'].mean():.3f} crimes/day")
print(f"  Probability calculated with Poisson distribution")
print(f"  Probability range: {df_map_source['crime_probability'].min()*100:.1f}% to {df_map_source['crime_probability'].max()*100:.1f}%")
print(f"  Cells with >50% probability: {(df_map_source['crime_probability'] > 0.5).sum():,}")

# Calculate map center from aggregated data
center_lat = df_map_source['lat'].median()
center_lon = df_map_source['lon'].median()

print(f"✓ Map center: ({center_lat:.4f}, {center_lon:.4f})")

Preparing forecast map data...
✓ Using aggregated cell data: 51,577 unique cells
  Average daily prediction: 0.027 crimes/day
  Probability calculated with Poisson distribution
  Probability range: 0.0% to 82.7%
  Cells with >50% probability: 10
✓ Map center: (-23.5647, -46.6396)


In [20]:
# Analyze crime types in predictions
print("Analyzing crime type predictions...")

# Check if RUBRICA column exists (crime type identifier)
if 'RUBRICA' in df_cell_agg.columns:
    # Group by cell and find the crime type with highest prediction
    df_cell_crime_type = df_cell_agg.groupby('h3_cell').apply(
        lambda x: pd.Series({
            'most_probable_crime': x.loc[x['y_pred'].idxmax(), 'RUBRICA'] if len(x) > 0 else 'Unknown',
            'max_prediction': x['y_pred'].max(),
            'total_prediction': x['predicted_total'].sum(),
            'lat': x['lat'].iloc[0],
            'lon': x['lon'].iloc[0],
            'crime_probability': x['crime_probability'].max(),
            'predicted_daily_avg': x['predicted_daily_avg'].sum(),
            'normalized_prob': x['normalized_prob'].max()
        })
    ).reset_index()
    
    # Map RUBRICA codes to readable crime type names
    crime_type_mapping = {
        'FURTO': 'Furto',
        'ROUBO': 'Roubo', 
        'HOMICIDIO': 'Homicídio',
        'LESAO CORPORAL': 'Lesão Corporal',
        'TRAFICO': 'Tráfico de Drogas',
        'AMEACA': 'Ameaça',
        'ESTUPRO': 'Estupro',
        'RECEPTACAO': 'Receptação',
        # Add more mappings as needed
    }
    
    # Apply mapping (if RUBRICA is already readable, this will just pass through)
    df_cell_crime_type['crime_type_display'] = df_cell_crime_type['most_probable_crime'].apply(
        lambda x: crime_type_mapping.get(x, x) if pd.notna(x) else 'Outros'
    )
    
    # Get unique crime types for layer filtering
    unique_crime_types = df_cell_crime_type['crime_type_display'].unique()
    
    print(f"✓ Crime type analysis complete")
    print(f"  Unique crime types: {len(unique_crime_types)}")
    print(f"  Crime type distribution:")
    for crime_type in sorted(unique_crime_types):
        count = (df_cell_crime_type['crime_type_display'] == crime_type).sum()
        pct = count / len(df_cell_crime_type) * 100
        print(f"    • {crime_type}: {count:,} cells ({pct:.1f}%)")
    
    # Use this aggregated data for mapping
    df_map_source = df_cell_crime_type.copy()
    
else:
    print("⚠️  RUBRICA column not found - crime type filtering unavailable")
    print("  Map will display without crime type information")
    unique_crime_types = []

print(f"\n✓ Map data prepared with crime type information")
print(f"  Total cells: {len(df_map_source):,}")
print(f"  Average prediction per cell: {df_map_source['total_prediction'].mean():.3f} crimes/week")

Analyzing crime type predictions...
✓ Crime type analysis complete
  Unique crime types: 5
  Crime type distribution:
    • Furto (art. 155): 11,833 cells (22.9%)
    • Furto qualificado (art. 155, §4o.): 9,508 cells (18.4%)
    • Homicídio simples (art. 121): 8,562 cells (16.6%)
    • OUTROS: 9,663 cells (18.7%)
    • Roubo (art. 157): 12,011 cells (23.3%)

✓ Map data prepared with crime type information
  Total cells: 51,577
  Average prediction per cell: 0.027 crimes/week
✓ Crime type analysis complete
  Unique crime types: 5
  Crime type distribution:
    • Furto (art. 155): 11,833 cells (22.9%)
    • Furto qualificado (art. 155, §4o.): 9,508 cells (18.4%)
    • Homicídio simples (art. 121): 8,562 cells (16.6%)
    • OUTROS: 9,663 cells (18.7%)
    • Roubo (art. 157): 12,011 cells (23.3%)

✓ Map data prepared with crime type information
  Total cells: 51,577
  Average prediction per cell: 0.027 crimes/week


In [21]:
# Create map with crime probability forecast (OPTIMIZED with better tile settings)
print("Creating crime probability forecast map...")
start_time = time.time()

# Create base map with high-resolution tile provider
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles=None,  # Don't use default tiles
    max_zoom=20,  # Allow deeper zoom
    min_zoom=10,  # Prevent zooming out too far
    zoom_control=True,
    prefer_canvas=True,  # Use canvas rendering for better performance
    max_bounds=True,  # Enable bounds restriction
    control_scale=True  # Add scale control
)

# Add high-resolution tile layers (these support zoom levels up to 20)
# CartoDB Voyager - best quality for high zoom
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png',
    name='CartoDB Voyager (HD)',
    max_zoom=20,
    max_native_zoom=20,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>',
    subdomains='abcd',
    overlay=False,
    control=True
).add_to(m)

# OpenStreetMap with proper max zoom
folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    name='OpenStreetMap',
    max_zoom=19,
    max_native_zoom=19,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
    overlay=False,
    control=True
).add_to(m)

# CartoDB Positron - clean style
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    name='CartoDB Positron',
    max_zoom=20,
    max_native_zoom=20,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>',
    subdomains='abcd',
    overlay=False,
    control=True
).add_to(m)

# CartoDB Dark Matter - dark style
folium.TileLayer(
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    name='CartoDB Dark Matter',
    max_zoom=20,
    max_native_zoom=20,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>',
    subdomains='abcd',
    overlay=False,
    control=True
).add_to(m)

# Color function (EXACT match with CrimeMap.jsx: light red -> dark red)
def get_red_gradient(normalized_value):
    """Return RGB color from light red to dark red based on normalized value (0-1).
    
    Matches CrimeMap.jsx implementation exactly:
    RGB: (255, 200, 200) -> (139, 0, 0)
    """
    r = int(255 - (116 * normalized_value))
    g = int(200 - (200 * normalized_value))
    b = int(200 - (200 * normalized_value))
    return f'#{r:02x}{g:02x}{b:02x}'

# Filter cells with meaningful crime probability (> 5%)
df_map = df_map_source[df_map_source['crime_probability'] > 0.05].copy()
print(f"Drawing {len(df_map):,} cells with probability > 5%...")

# Calculate percentile thresholds for legend
p33 = df_map['crime_probability'].quantile(0.33)
p67 = df_map['crime_probability'].quantile(0.67)
min_prob = df_map['crime_probability'].min()
max_prob = df_map['crime_probability'].max()

# Build crime type layers if available
crime_type_layers = {}
if len(unique_crime_types) > 0:
    print(f"\nCreating {len(unique_crime_types)} crime type layers for filtering...")
    for crime_type in unique_crime_types:
        crime_type_layers[crime_type] = []

# Get model error metrics for margin calculation
model_mae_test = metadata['metrics']['test']['mae']  # Mean Absolute Error
model_rmse_test = metadata['metrics']['test']['rmse']  # Root Mean Squared Error
model_r2_test = metadata['metrics']['test']['r2']  # R² Score

# Calculate error margins (for weekly predictions)
# MAE is for daily predictions, so multiply by 7 for weekly
weekly_mae = model_mae_test * 7
weekly_rmse = model_rmse_test * 7

print(f"\n📊 Model Error Metrics (from test set):")
print(f"  • MAE (daily): {model_mae_test:.4f} crimes/day")
print(f"  • MAE (weekly): {weekly_mae:.4f} crimes/week")
print(f"  • RMSE (weekly): {weekly_rmse:.4f} crimes/week")
print(f"  • R² Score: {model_r2_test:.4f}")

# Vectorize boundary and color computation
df_map['fill_color'] = df_map['normalized_prob'].apply(get_red_gradient)
df_map['boundary'] = df_map['h3_cell'].apply(lambda x: helpers.h3_to_boundary(x))

# Build features with crime type information
all_features = []

for idx, row in df_map.iterrows():
    # Convert boundary to GeoJSON format (lon, lat)
    coords = [[[lon, lat] for lat, lon in row['boundary']]]
    
    # Calculate prediction interval (±MAE for 68% confidence, ±2*MAE for ~95%)
    prediction_lower = max(0, row['total_prediction'] - weekly_mae)
    prediction_upper = row['total_prediction'] + weekly_mae
    prediction_lower_95 = max(0, row['total_prediction'] - 2*weekly_mae)
    prediction_upper_95 = row['total_prediction'] + 2*weekly_mae
    
    # Get crime type info if available
    crime_type_info = ""
    crime_type_display = "N/A"
    if 'crime_type_display' in row:
        crime_type_display = row['crime_type_display']
        crime_type_info = (
            f"<hr style='margin: 5px 0; border: none; border-top: 1px solid #ddd;'/>"
            f"<b>Tipo de Crime Mais Provável:</b> <span style='color: #e74c3c; font-weight: bold;'>{crime_type_display}</span><br/>"
        )
    
    # Popup content (Portuguese - forecast with error margin and crime type)
    popup_html = (
        f"<div style='font-family: Arial, sans-serif; font-size: 12px;'>"
        f"<b style='font-size: 14px;'>Previsão de Ocorrência de Crime</b><br/>"
        f"<hr style='margin: 5px 0; border: none; border-top: 1px solid #ddd;'/>"
        f"<b>Célula:</b> {row['h3_cell'][:8]}...<br/>"
        f"<b>Probabilidade:</b> {row['crime_probability']*100:.1f}%<br/>"
        f"<b>Período:</b> 7 dias<br/>"
        f"{crime_type_info}"
        f"<hr style='margin: 5px 0; border: none; border-top: 1px solid #ddd;'/>"
        f"<b>Média Diária Prevista:</b> {row['predicted_daily_avg']:.3f}/dia<br/>"
        f"<b>Total Previsto (Semanal):</b> {row['total_prediction']:.2f} crimes<br/>"
        f"<hr style='margin: 5px 0; border: none; border-top: 1px solid #ddd;'/>"
        f"<b>Margem de Erro (MAE):</b> ±{weekly_mae:.2f} crimes<br/>"
        f"<b>Intervalo 68%:</b> {prediction_lower:.2f} - {prediction_upper:.2f}<br/>"
        f"<b>Intervalo 95%:</b> {prediction_lower_95:.2f} - {prediction_upper_95:.2f}<br/>"
        f"<hr style='margin: 5px 0; border: none; border-top: 1px solid #aaa;'/>"
        f"<span style='font-size: 10px; color: #666;'>Modelo R²: {model_r2_test:.3f} | MAE Teste: {model_mae_test:.4f}</span>"
        f"</div>"
    )
    
    feature = {
        'type': 'Feature',
        'geometry': {
            'type': 'Polygon',
            'coordinates': coords
        },
        'properties': {
            'fillColor': row['fill_color'],
            'color': row['fill_color'],
            'weight': 1,
            'fillOpacity': 0.6,
            'popup': popup_html,
            'tooltip': f"{crime_type_display}: {row['crime_probability']*100:.1f}%",
            'crime_type': crime_type_display
        }
    }
    
    all_features.append(feature)
    
    # Add to crime-specific layer if available
    if len(unique_crime_types) > 0 and 'crime_type_display' in row:
        crime_type_layers[crime_type_display].append(feature)

# Create single GeoJSON layer with optimized settings
geojson_data = {
    'type': 'FeatureCollection',
    'features': all_features
}

# Add main GeoJSON layer (all crime types)
folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        'fillColor': feature['properties']['fillColor'],
        'color': feature['properties']['fillColor'],
        'weight': 1,
        'fillOpacity': 0.6,
        'smoothFactor': 0.5  # Lower smooth factor for sharper rendering at high zoom
    },
    popup=folium.GeoJsonPopup(fields=['popup'], labels=False),
    tooltip=folium.GeoJsonTooltip(fields=['tooltip'], labels=False, sticky=True),
    smooth_factor=0.5,  # Lower smooth factor for better detail
    name='Todos os Tipos de Crime',
    overlay=True,
    control=True,
    show=True
).add_to(m)

# Add crime-specific layers if available
if len(unique_crime_types) > 0:
    print(f"\nAdding {len(crime_type_layers)} filterable crime type layers...")
    
    # Define colors for different crime types
    crime_type_colors = {
        'Furto': '#FF6B6B',       # Red
        'Roubo': '#FFA500',       # Orange
        'Homicídio': '#8B0000',   # Dark red
        'Lesão Corporal': '#FFD700', # Gold
        'Tráfico de Drogas': '#9370DB', # Purple
        'Ameaça': '#FF69B4',      # Pink
        'Estupro': '#DC143C',     # Crimson
        'Receptação': '#FF8C00',  # Dark orange
    }
    
    # Sort crime types by frequency (most to least), with "Outros" always last
    crime_type_counts = [(ct, len(features)) for ct, features in crime_type_layers.items()]
    crime_type_counts_sorted = sorted(
        [item for item in crime_type_counts if item[0] != 'Outros'],
        key=lambda x: x[1],
        reverse=True
    )
    # Add "Outros" at the end if it exists
    if 'Outros' in crime_type_layers:
        crime_type_counts_sorted.append(('Outros', len(crime_type_layers['Outros'])))
    
    print(f"  Crime types ordered by frequency:")
    for crime_type, count in crime_type_counts_sorted:
        print(f"    {count:,} cells → {crime_type}")
    
    # Add layers in sorted order
    for crime_type, _ in crime_type_counts_sorted:
        features = crime_type_layers[crime_type]
        if len(features) > 0:
            crime_geojson = {
                'type': 'FeatureCollection',
                'features': features
            }
            
            # Get color for this crime type (default to gray if not mapped)
            layer_color = crime_type_colors.get(crime_type, '#808080')
            
            folium.GeoJson(
                crime_geojson,
                style_function=lambda feature, color=layer_color: {
                    'fillColor': feature['properties']['fillColor'],
                    'color': feature['properties']['fillColor'],
                    'weight': 1,
                    'fillOpacity': 0.6,
                    'smoothFactor': 0.5
                },
                popup=folium.GeoJsonPopup(fields=['popup'], labels=False),
                tooltip=folium.GeoJsonTooltip(fields=['tooltip'], labels=False, sticky=True),
                smooth_factor=0.5,
                name=f'🔍 {crime_type} ({len(features)} células)',
                overlay=True,
                control=True,
                show=False  # Start hidden - user can toggle on
            ).add_to(m)
            
            print(f"  • {crime_type}: {len(features)} cells")

# Add layer control to toggle between tile layers and crime types (COLLAPSED by default)
folium.LayerControl(position='topright', collapsed=True).add_to(m)

# Enhanced legend - forecast with crime type filtering (COLLAPSIBLE, BOTTOM-RIGHT)
legend_html = f'''
<div id="legend-container" style="position: fixed; 
     bottom: 10px; right: 10px; width: 340px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:13px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.3)">
     
     <div id="legend-header" style="padding: 10px 12px; background-color: #f0f0f0; cursor: pointer; 
          border-bottom: 2px solid #ddd; display: flex; justify-content: space-between; align-items: center;"
          onclick="toggleLegend()">
          <b style="margin:0">Previsão de Probabilidade de Crime</b>
          <span id="legend-toggle" style="font-size: 18px; font-weight: bold;">−</span>
     </div>
     
     <div id="legend-content" style="padding: 12px; display: block;">
     <p style="margin:0 0 8px 0"><b>Escala de Probabilidade:</b></p>
     
     <p style="margin:5px 0"><span style="background-color:#ffc8c8; padding:2px 10px">▮</span> 
        Baixa ({min_prob*100:.1f}% - {p33*100:.1f}%)</p>
     <p style="margin:5px 0"><span style="background-color:#ff6464; padding:2px 10px">▮</span> 
        Média ({p33*100:.1f}% - {p67*100:.1f}%)</p>
     <p style="margin:5px 0"><span style="background-color:#8b0000; padding:2px 10px">▮</span> 
        Alta ({p67*100:.1f}% - {max_prob*100:.1f}%)</p>
     
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ddd">
     
     <p style="margin:5px 0; font-size:11px; color:#333"><b>Período de Previsão:</b> {forecast_period}</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Método:</b> Poisson (P ≥ 1 crime/dia)</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Agregação:</b> Semanal (H3 Res 10)</p>
     <p style="margin:5px 0; font-size:11px; color:#666"><b>Filtro:</b> Probabilidade > 10%</p>
     
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ddd">
     
     <p style="margin:5px 0; font-size:11px; color:#333"><b>Margem de Erro (MAE):</b> ±{weekly_mae:.2f} crimes/semana</p>
     <p style="margin:5px 0; font-size:11px; color:#333"><b>R² do Modelo:</b> {model_r2_test:.3f} ({model_r2_test*100:.1f}% de precisão)</p>
     
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ddd">
     
     <p style="margin:5px 0; font-size:11px; color:#2ecc71; font-weight:bold">🔍 Use o controle de camadas (canto superior direito)</p>
     <p style="margin:5px 0; font-size:10px; color:#666; font-style:italic">Filtre por tipo de crime específico</p>
     
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ddd">
     
     <p style="margin:5px 0; font-size:11px; color:#e74c3c; font-weight:bold">⚠️ Apenas Previsões (sem dados reais)</p>
     <p style="margin:5px 0; font-size:10px; color:#999; font-style:italic">Clique nas células para ver intervalos de confiança e tipo de crime</p>
     </div>
</div>

<script>
function toggleLegend() {{
    var content = document.getElementById('legend-content');
    var toggle = document.getElementById('legend-toggle');
    if (content.style.display === 'none') {{
        content.style.display = 'block';
        toggle.innerHTML = '−';
    }} else {{
        content.style.display = 'none';
        toggle.innerHTML = '+';
    }}
}}
</script>
'''
m.get_root().html.add_child(folium.Element(legend_html))
elapsed = time.time() - start_time
print(f"✓ Forecast map created in {elapsed:.2f} seconds!")
print(f"\nMap Statistics:")
print(f"  • Cells displayed: {len(df_map):,}")
print(f"  • Probability range: {min_prob*100:.1f}% to {max_prob*100:.1f}%")
print(f"  • Percentile thresholds:")
print(f"    - 33rd percentile (Baixa/Média): {p33*100:.1f}%")
print(f"    - 67th percentile (Média/Alta): {p67*100:.1f}%")
print(f"  • High probability (>{p67*100:.1f}%): {(df_map['crime_probability'] > p67).sum()}")
print(f"  • Medium probability ({p33*100:.1f}%-{p67*100:.1f}%): {((df_map['crime_probability'] >= p33) & (df_map['crime_probability'] <= p67)).sum()}")
print(f"  • Low probability (<{p33*100:.1f}%): {(df_map['crime_probability'] < p33).sum()}")

print(f"\n📊 Model Accuracy & Error Margins:")
print(f"  • R² Score: {model_r2_test:.4f} ({model_r2_test*100:.2f}% explained variance)")
print(f"  • MAE (weekly): ±{weekly_mae:.3f} crimes")
print(f"  • RMSE (weekly): ±{weekly_rmse:.3f} crimes")
print(f"  • 68% confidence: prediction ± {weekly_mae:.2f}")
print(f"  • 95% confidence: prediction ± {2*weekly_mae:.2f}")

if len(unique_crime_types) > 0:
    print(f"\n🔍 Crime Type Filtering:")
    print(f"  • {len(unique_crime_types)} filterable crime type layers created")
    print(f"  • Use layer control (top-right) to filter by specific crime types")
    print(f"  • Each cell shows the most probable crime type in popup")

print(f"\n✓ Forecast map with error margins and crime type filtering")

print(f"  Click cells for detailed predictions including crime type")

m

Creating crime probability forecast map...
Drawing 1,096 cells with probability > 5%...

Creating 5 crime type layers for filtering...

📊 Model Error Metrics (from test set):
  • MAE (daily): 0.0039 crimes/day
  • MAE (weekly): 0.0276 crimes/week
  • RMSE (weekly): 0.3941 crimes/week
  • R² Score: 0.9324

Adding 5 filterable crime type layers...
  Crime types ordered by frequency:
    604 cells → Roubo (art. 157)
    350 cells → Furto (art. 155)
    89 cells → OUTROS
    50 cells → Furto qualificado (art. 155, §4o.)
    3 cells → Homicídio simples (art. 121)
  • Roubo (art. 157): 604 cells
  • Furto (art. 155): 350 cells
  • OUTROS: 89 cells
  • Furto qualificado (art. 155, §4o.): 50 cells
  • Homicídio simples (art. 121): 3 cells
✓ Forecast map created in 0.11 seconds!

Map Statistics:
  • Cells displayed: 1,096
  • Probability range: 5.6% to 82.7%
  • Percentile thresholds:
    - 33rd percentile (Baixa/Média): 13.9%
    - 67th percentile (Média/Alta): 14.6%
  • High probability (>14.

In [ ]:
# Export Forecast Panel to Panels Folder (Sparse Format)
print("💾 Exporting forecast panel to panels folder...")

# Map RUBRICA to readable crime_type if RUBRICA exists
if 'RUBRICA' in df_predicted_panel.columns:
    print("  🔍 Mapping RUBRICA codes to readable crime types...")
    
    # Crime type mapping (same as used for notebook map visualization)
    crime_type_mapping = {
        'FURTO': 'Furto',
        'ROUBO': 'Roubo', 
        'HOMICIDIO': 'Homicídio',
        'LESAO CORPORAL': 'Lesão Corporal',
        'TRAFICO': 'Tráfico de Drogas',
        'AMEACA': 'Ameaça',
        'ESTUPRO': 'Estupro',
        'RECEPTACAO': 'Receptação',
    }
    
    df_predicted_panel['crime_type'] = df_predicted_panel['RUBRICA'].apply(
        lambda x: crime_type_mapping.get(str(x).upper(), str(x)) if pd.notna(x) else 'Outros'
    )
    
    # Drop RUBRICA column after mapping (keep only crime_type)
    df_predicted_panel = df_predicted_panel.drop(columns=['RUBRICA'])
    
    crime_type_counts = df_predicted_panel['crime_type'].value_counts()
    print(f"  ✓ Crime types mapped: {len(crime_type_counts)} unique types")
    print(f"    Top 3: {', '.join([f'{ct} ({cnt})' for ct, cnt in crime_type_counts.head(3).items()])}")
else:
    print("  ⚠️  RUBRICA column not found - crime type filtering will not be available!")
    print("     Frontend will only show 'Todos os Tipos' layer")

# Verify panel is sparse (one row per H3 cell)
panel_cells = df_predicted_panel['h3_cell'].nunique()
panel_rows = len(df_predicted_panel)

if panel_cells == panel_rows:
    print(f"  ✓ Panel is sparse: {panel_rows:,} rows = {panel_cells:,} unique cells")
else:
    print(f"  ⚠️  Panel is not sparse: {panel_rows:,} rows vs {panel_cells:,} unique cells")
    print("    Creating sparse version...")
    # If not sparse, aggregate to one row per cell (take max prediction per cell)
    df_predicted_panel = df_predicted_panel.loc[df_predicted_panel.groupby('h3_cell')['predicted_total'].idxmax()].copy()
    print(f"    ✓ Sparse panel created: {len(df_predicted_panel):,} rows")

# Calculate predicted week of the year
predicted_week = forecast_start.isocalendar()[1]
predicted_year = forecast_start.year

# Auto-increment forecast number based on existing folders
panels_dir = project_root / "panels"
panels_dir.mkdir(parents=True, exist_ok=True)

# Find the next available forecast number
existing_forecasts = sorted(panels_dir.glob("PrepolForecast_*"))
if existing_forecasts:
    # Extract numbers from existing folders (e.g., "PrepolForecast_02" -> 2)
    existing_numbers = []
    for folder in existing_forecasts:
        try:
            num = int(folder.name.split('_')[-1])
            existing_numbers.append(num)
        except ValueError:
            continue
    next_forecast_num = max(existing_numbers) + 1 if existing_numbers else 1
else:
    next_forecast_num = 1

# Create filename and folder name with auto-incremented number
forecast_name = f"PrepolForecast_{next_forecast_num:02d}"
folder_name = forecast_name
parquet_filename = f"{forecast_name}.parquet"
metadata_filename = f"{forecast_name}_metadata.json"

print(f"\n📁 Auto-incrementing forecast number to: {next_forecast_num:02d}")
print(f"   Folder: {forecast_name}")

# Create forecast subfolder path
forecast_dir = panels_dir / folder_name

# Create directory structure
forecast_dir.mkdir(parents=True, exist_ok=True)

# Export parquet file (all columns including crime_type)
parquet_path = forecast_dir / parquet_filename
df_predicted_panel.to_parquet(parquet_path, index=False, compression='snappy')

print(f"\n✓ Parquet exported: {parquet_path}")
print(f"  Rows: {len(df_predicted_panel):,}")
print(f"  Size: {parquet_path.stat().st_size / 1024:.1f} KB")
print(f"  Columns: {list(df_predicted_panel.columns)}")

# Create comprehensive metadata
metadata = {
    "forecast_info": {
        "generated_at": datetime.now().isoformat(),
        "forecast_period": {
            "week_of_year": predicted_week,
            "year": predicted_year,
            "start_date": target_start,
            "end_date": target_end,
            "duration_days": 7,
            "frequency": "weekly"
        },
        "panel_type": "sparse_forecast",
        "h3_resolution": config.H3_RES
    },
    "model_info": {
        "model_file": model_path.name,
        "test_r2": metadata['metrics']['test']['r2'],
        "test_mae": metadata['metrics']['test']['mae'],
        "test_rmse": metadata['metrics']['test']['rmse'],
        "feature_columns": feature_cols
    },
    "panel_statistics": {
        "total_cells": int(len(df_predicted_panel)),
        "total_predicted_crimes": float(df_predicted_panel['predicted_total'].sum()),
        "mean_daily_prediction": float(df_predicted_panel['predicted_daily_avg'].mean()),
        "median_daily_prediction": float(df_predicted_panel['predicted_daily_avg'].median()),
        "max_daily_prediction": float(df_predicted_panel['predicted_daily_avg'].max()),
        "cells_with_predictions": int((df_predicted_panel['predicted_total'] > 0).sum()),
        "probability_stats": {
            "mean": float(df_predicted_panel['crime_probability'].mean()),
            "median": float(df_predicted_panel['crime_probability'].median()),
            "min": float(df_predicted_panel['crime_probability'].min()),
            "max": float(df_predicted_panel['crime_probability'].max()),
            "cells_high_risk_80pct": int((df_predicted_panel['crime_probability'] > 0.8).sum()),
            "cells_high_risk_50pct": int((df_predicted_panel['crime_probability'] > 0.5).sum())
        }
    },
    "data_structure": {
        "columns": list(df_predicted_panel.columns),
        "dtypes": {col: str(dtype) for col, dtype in df_predicted_panel.dtypes.items()},
        "sparsity": "one_row_per_h3_cell",
        "has_crime_type": 'crime_type' in df_predicted_panel.columns
    },
    "export_info": {
        "parquet_file": parquet_filename,
        "metadata_file": metadata_filename,
        "folder": folder_name,
        "compression": "snappy",
        "format_version": "1.0"
    },
    "usage_notes": {
        "probability_calculation": "Poisson distribution P(≥1 crime/day) = 1 - e^(-daily_avg)",
        "daily_conversion": "weekly_predictions / 7 = daily_avg",
        "matches_backend": "Compatible with server.py and CrimeMap.jsx",
        "sparse_format": "One row per H3 cell with predictions",
        "crime_type_filtering": "Column 'crime_type' used for map filtering layers" if 'crime_type' in df_predicted_panel.columns else "Crime type filtering not available"
    }
}

# Export metadata JSON
metadata_path = forecast_dir / metadata_filename
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Metadata exported: {metadata_path}")
print(f"  Size: {metadata_path.stat().st_size / 1024:.1f} KB")

print(f"\n📁 Forecast panel exported to: {forecast_dir}")
print(f"  📄 {parquet_filename}")
print(f"  📋 {metadata_filename}")
print(f"\n🎯 Forecast Summary:")
print(f"  • Week {predicted_week} of {predicted_year}")
print(f"  • {len(df_predicted_panel):,} H3 cells with predictions")
print(f"  • Total predicted crimes: {df_predicted_panel['predicted_total'].sum():.0f}")
print(f"  • Average daily probability: {df_predicted_panel['crime_probability'].mean()*100:.1f}%")
print(f"  • High-risk cells (>50%): {(df_predicted_panel['crime_probability'] > 0.5).sum():,}")

if 'crime_type' in df_predicted_panel.columns:
    unique_crime_types = df_predicted_panel['crime_type'].nunique()
    print(f"  • Crime types: {unique_crime_types} unique types for filtering")
    print(f"    ✅ Frontend crime type layers will be available!")
else:
    print(f"  ⚠️  No crime_type column - frontend will show 'Todos os Tipos' only")

print(f"\n✅ Forecast panel export complete!")
print(f"\n📤 Next steps:")
print(f"  1. Run import_forecast_to_mongodb.py to upload to production")
print(f"  2. Set USE_MONGODB_FORECAST = True in server.py")
print(f"  3. Frontend will automatically show crime type filters!")


💾 Exporting forecast panel to panels folder...
  🔍 Mapping RUBRICA codes to readable crime types...
  ✓ Crime types mapped: 5 unique types
    Top 3: Roubo (art. 157) (12011), Furto (art. 155) (11833), OUTROS (9663)
  ✓ Panel is sparse: 51,577 rows = 51,577 unique cells

📁 Auto-incrementing forecast number to: 03
   Folder: PrepolForecast_03

✓ Parquet exported: d:\BackupSupremo\Work\prepol-project\panels\PrepolForecast_03\PrepolForecast_03.parquet
  Rows: 51,577
  Size: 1634.6 KB
  Columns: ['h3_cell', 'period', 'predicted_daily_avg', 'predicted_total', 'crime_probability', 'lat', 'lon', 'n_days', 'crime_type']
✓ Metadata exported: d:\BackupSupremo\Work\prepol-project\panels\PrepolForecast_03\PrepolForecast_03_metadata.json
  Size: 2.5 KB

📁 Forecast panel exported to: d:\BackupSupremo\Work\prepol-project\panels\PrepolForecast_03
  📄 PrepolForecast_03.parquet
  📋 PrepolForecast_03_metadata.json

🎯 Forecast Summary:
  • Week 2 of 1970
  • 51,577 H3 cells with predictions
  • Total pred